# 🔗 WitnessChain — Demo Notebook

**Trauma-informed, multilingual testimony infrastructure powered by Gemma 4**

![Domain](https://img.shields.io/badge/Domain-Safety%20%26%20Trust-red)
![Unsloth](https://img.shields.io/badge/Fine--tuned%20with-Unsloth-green)

This notebook:
1. Installs all dependencies
2. Authenticates with HuggingFace
3. Loads Gemma 4 (27B or 12B, based on available VRAM)
4. Launches the WitnessChain Gradio app with a public URL

**Runtime:** GPU → A100 (preferred) or T4 (fallback to 12B model)

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers>=4.40.0 accelerate>=0.27.0 bitsandbytes>=0.43.0 \
    gradio>=4.20.0 langdetect>=1.0.9 fpdf2>=2.7.0 python-docx>=1.1.0 \
    unsloth[colab-new]>=2024.3 torch>=2.2.0 datasets>=2.18.0 trl>=0.8.0 \
    sentencepiece>=0.1.99

## Step 2: Authenticate with HuggingFace

Set your HuggingFace token as a Colab secret named `HF_TOKEN`.

Go to: **Runtime → Secrets → Add `HF_TOKEN`**

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

!huggingface-cli login --token $HF_TOKEN
print('✅ HuggingFace authentication complete.')

## Step 3: Clone WitnessChain Repository

If you're running this from the repo already, skip this cell.

In [ ]:
# If running from a cloned repo, uncomment and update the URL:
# !git clone https://github.com/<your-github-username>/witnesschain.git
# %cd witnesschain

# If running from repo root:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"\u2705 Path configured: {repo_root}")


## Step 4: Check GPU & VRAM

WitnessChain auto-selects the optimal model:
- **A100 (40GB+)** → Gemma 4 27B (4-bit quantised)
- **T4 (16GB)** → Gemma 4 12B (4-bit quantised)

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✅ GPU detected: {gpu_name}')
    print(f'   VRAM: {vram_gb:.1f} GB')
    print(f'   Model: {"27B" if vram_gb >= 35 else "12B"} will be loaded')
else:
    print('⚠️  No GPU detected. Please enable GPU runtime.')
    print('   Go to: Runtime → Change runtime type → GPU')


## Step 5: Launch WitnessChain

This cell:
1. Loads Gemma 4 with 4-bit quantisation
2. Initialises the TRUST framework and all engines
3. Launches the Gradio app with a **public URL** (`share=True`)

The public URL will appear below — click it to open the demo.

In [ ]:
import sys, os, torch

# Robust path setup — works regardless of notebook launch directory
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Reproducibility seed
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

from app import init_system, build_app

print("[WitnessChain] Initialising system...")
init_system()

print("[WitnessChain] Building Gradio app...")
demo = build_app()

print("[WitnessChain] Launching with share=True...")
demo.launch(share=True, debug=False)


## Demo Script

For hackathon video / judging:

1. Open the Gradio public URL above
2. Go to **New Testimony** tab
3. Type in Arabic: `أنا رأيت الجنود يحرقون القرية في 12 مارس`
4. Observe TRUST-governed response (validation → one question)
5. Complete 3-4 turn interview
6. Switch to **Cross-Reference** tab → click **Analyse Corroboration**
7. Observe token counter + corroboration output
8. Go to **Case Report** → download PDF
9. Check **Ethical Audit Log** → every prompt visible
10. Toggle model between Base and Fine-tuned to show improvement